# DER-only RGAT: Accuracy and F1 Evaluation

This notebook adds **one RGAT model** to the existing DER graph-classification pipeline. It keeps the same annotation-to-graph construction and evaluates three existing node encoders: **SBERT, BERT-iso and BERT-ctx**.

**Outputs:** accuracy, macro-F1 and weighted-F1, reported across repeated cross-validation runs. No explainability analyses or additional graph architectures are run here.

**Protocol:** DER nodes and DER-to-DER relations only; the `Sentence` node and its `contains` edges are excluded. Relations are collapsed into the established DER relation families to avoid a very sparse edge vocabulary. CV is plain stratified 5-fold on both datasets: the annotation files carry no `template_id`, so a grouped split offered no real minimal-pair protection over stratified and was dropped (see README "Known gaps" for the decision record).

Run cells from top to bottom from the repo root. The four JSON files listed in the data section are expected in `data/`.


## 0. Setup and configuration

In [ ]:
# Install dependencies (Colab). spaCy is optional (identity lemmatiser falls back to plural stripping).
%pip install -q transformers sentence-transformers torch-geometric scikit-learn pandas numpy networkx matplotlib


In [ ]:
import copy, random, re, itertools, json, warnings, gc, time, os
from collections import Counter, defaultdict
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch_geometric.data import HeteroData, Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import RGATConv, global_mean_pool
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore", category=UserWarning)

# Full = repeated 5-fold CV across both evaluation plans (InDiBias/stratified, IndiCasa/stratified).
# "Grouped" CV was dropped: it was meant to protect InDiBias minimal pairs via template_id, but
# the annotation files have no template_id, so it offered no real protection over stratified.
# Quick = one repeat on InDiBias/stratified only, for a fast smoke test.
RUN_PROFILE = "full"  # change to "quick" for a fast smoke test
PROFILES = {
    "quick": dict(n_repeats=1, analysis_plan=[("InDiBias", "stratified")]),
    "full": dict(n_repeats=3, analysis_plan=[("InDiBias", "stratified"),
                                             ("IndiCasa", "stratified")]),
}
P = PROFILES[RUN_PROFILE]
SEED = 42
N_REPEATS = P["n_repeats"]
SEEDS = [SEED + 100 * r for r in range(N_REPEATS)]
EVAL_PLAN = P["analysis_plan"]

# Model/training settings; held constant across encoders.
TRAIN_ON_GPU = True
ENCODER_FP16 = True
ENC_BATCH = 64
BATCH_SIZE = 16
EPOCHS = 40
LR = 1e-3
# Compare a plain baseline against the current regularised setup.
# Base: no dropout and no weight decay.
# Reg: original settings (dropout=0.3, weight_decay=1e-4).
HIDDEN = 64
RGAT_HEADS = 2
CONFIGS = {
    "base": {"dropout": 0.0, "weight_decay": 0.0},
    "reg":  {"dropout": 0.3, "weight_decay": 1e-4},
}
RELATION_MODES = ["collapsed", "raw"]  # compare canonical collapsed relations with original raw relations

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_device = device if TRAIN_ON_GPU else torch.device("cpu")
os.makedirs("results", exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"profile={RUN_PROFILE} | repeats={N_REPEATS} | seeds={SEEDS} | device={device}")
if device.type == "cuda": print("GPU:", torch.cuda.get_device_name(0))


## 1. Load the existing annotated datasets

Expected files in `data/` (relative to repo root):
- `indibias_pos.json`
- `indibias_neg.json`
- `indicasa_pos.json`
- `indicasa_neg.json`

If your files are in Google Drive or a different directory, change `DATA_DIR` below. Positive examples receive label `1` (stereotype); negative examples receive label `0` (anti-stereotype), matching the existing pipeline.


In [ ]:
import json
from pathlib import Path

DATA_DIR = Path("data")  # relative to repo root

with open(DATA_DIR / "indibias_pos.json", "r", encoding="utf-8") as f:
    EXAMPLE_ANNOTATIONS_POS_INDIBIAS = json.load(f)

with open(DATA_DIR / "indibias_neg.json", "r", encoding="utf-8") as f:
    EXAMPLE_ANNOTATIONS_NEG_INDIBIAS = json.load(f)

with open(DATA_DIR / "indicasa_pos.json", "r", encoding="utf-8") as f:
    EXAMPLE_ANNOTATIONS_POS_INDICASA = json.load(f)

with open(DATA_DIR / "indicasa_neg.json", "r", encoding="utf-8") as f:
    EXAMPLE_ANNOTATIONS_NEG_INDICASA = json.load(f)

In [ ]:
print("InDiBias positive:", len(EXAMPLE_ANNOTATIONS_POS_INDIBIAS))
print("InDiBias negative:", len(EXAMPLE_ANNOTATIONS_NEG_INDIBIAS))
print("IndiCasa positive:", len(EXAMPLE_ANNOTATIONS_POS_INDICASA))
print("IndiCasa negative:", len(EXAMPLE_ANNOTATIONS_NEG_INDICASA))

InDiBias positive: 243
InDiBias negative: 83
IndiCasa positive: 243
IndiCasa negative: 83


In [ ]:
DATASETS = {
    "InDiBias": dict(pos=EXAMPLE_ANNOTATIONS_POS_INDIBIAS, neg=EXAMPLE_ANNOTATIONS_NEG_INDIBIAS),
    "IndiCasa": dict(pos=EXAMPLE_ANNOTATIONS_POS_INDICASA, neg=EXAMPLE_ANNOTATIONS_NEG_INDICASA),
}
for name, d in DATASETS.items():
    print(f"{name}: {len(d['pos'])} stereotype / {len(d['neg'])} anti-stereotype")

InDiBias: 243 stereotype / 83 anti-stereotype
IndiCasa: 243 stereotype / 83 anti-stereotype


## 2. Build the same DER sentence graphs

In [ ]:
# Identity normalisation ("Hindus" -> "hindu", "OBC family" -> "obc", ...)
try:
    import spacy
    _nlp = spacy.load("en_core_web_sm")
except (ImportError, OSError):
    _nlp = None          # falls back to simple plural stripping
print("spaCy lemmatiser:", "on" if _nlp is not None else "off (fallback)")

_IDENTITY_MODIFIERS = frozenset({
    "family", "families", "group", "community", "commnunity",
    "people", "person", "man", "woman", "child", "children"
})
_PLURAL_OVERRIDES = {
    "hindus": "hindu", "muslims": "muslim", "vaishyas": "vaishya", "brahmins": "brahmin",
    "dalits": "dalit", "obcs": "obc", "christians": "christian", "sikhs": "sikh",
    "jains": "jain", "buddhists": "buddhist",
}
_COMPOUND_PATTERNS = [
    (r"\b(obc|hindu|muslim|brahmin|dalit|vaishya|christian|sikh|jain)\s+(family|group|community|people)\b", r"\1"),
    (r"\b(hindu|muslim|brahmin|dalit|vaishya)\s+(man|woman|person|child)\b", r"\1"),
]


def _strip_plural(text):
    return text[:-1] if text.endswith("s") and not text.endswith(("ss", "us", "is")) else text


def _normalize_identity(span: str) -> str:
    text = span.lower().strip()
    for pattern, repl in _COMPOUND_PATTERNS:
        text = re.sub(pattern, repl, text, flags=re.IGNORECASE)
    words = text.split()
    if len(words) > 1 and words[-1] in _IDENTITY_MODIFIERS:
        text = " ".join(words[:-1])
    if text in _PLURAL_OVERRIDES:
        text = _PLURAL_OVERRIDES[text]
    if _nlp is not None:
        try:
            text = " ".join(t.lemma_ for t in _nlp(text))
        except Exception:
            text = _strip_plural(text)
    else:
        text = _strip_plural(text)
    return text.strip()

spaCy lemmatiser: on


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MAPPING IDENTITY TO CATEGORY
# ─────────────────────────────────────────────────────────────────────────────

_IDENTITY_TO_CATEGORY = {
    # Religion
    "hindu": "Religion",
    "muslim": "Religion",
    "christian": "Religion",
    "sikh": "Religion",
    "jain": "Religion",
    "buddhist": "Religion",
    "islam" :  "Religion",
    "christianity": "Religion",
    "hinduism": "Religion",
    "sikhism" : "Religion",
    "parsi" : "Religion",
    "parsis": "Religion",
    "buddhism": "Religion",
    "jainism": "Religion",

    # Caste
    "brahmin": "Caste",
    "vaishya": "Caste",
    "dalit": "Caste",
    "obc": "Caste",
    "sc": "Caste",
    "st": "Caste",
    "kshatriya": "Caste",
    "shudra": "Caste",
    "sc / st": "Caste",
    "kaystha": "Caste",
    "bania": "Caste",
    "kurmi": "Caste",
    "jat": "Caste",
    "rajput": "Caste",
    "adivasi": "Caste",
    "kayast": "Caste",
    "general": "Caste",
    "forward class": "Caste",
    "artisan": "Caste",
    "merchant": "Caste",
    "trader": "Caste",
    "laborer": "Caste",
    "peasant": "Caste",

    # Gender
    "women": "Gender",
    "man": "Gender",
    "woman": "Gender",
    "female": "Gender",
    "male": "Gender",
    "boy": "Gender",
    "girl": "Gender",
    "girls": "Gender",
    "boys": "Gender",
    "men": "Gender",
    "transgender male": "Gender",
    "lesbian": "Gender"
}

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 1. RELATION EXTRACTION (unchanged structure)
# ══════════════════════════════════════════════════════════════════════════════

class RuleBasedRelations:
    """
    Converts a DER annotation dict into a flat list of canonical edges.
    Structure unchanged, only the node extraction uses new normalization.
    """

    # ── Node-type labels ──────────────────────────────────────────────────
    IDENTITY_NODE   = "Identity Dimensions"
    PROFESSION_NODE = "Profession"
    TRIGGER_NODE    = "Contextual Trigger"
    FRAME_NODE      = "Situational Frame"

    # ── Canonical relation-type vocabulary ────────────────────────────────
    REL_IDENTITY_INTERSECTION        = "identity_intersection"
    REL_IDENTITY_ROLE_ASSOCIATION    = "identity_role_association"
    REL_CHARACTERIZED_AS             = "characterized_as"
    REL_IDENTITY_FRAME_ASSOCIATION= "identity_frame_association"

    # ── Copula / auxiliary / modal verbs ──────────────────────────────────
    PREDICATE_LEAD_WORDS: frozenset = frozenset({
        "is", "are", "was", "were", "be", "been", "being",
        "do", "does", "did",
        "has", "have", "had",
        "can", "could", "will", "would", "shall", "should",
        "may", "might", "must",
        "seem", "seems", "seemed",
        "appear", "appears", "appeared",
        "become", "becomes", "became",
        "get", "gets", "got",
        "remain", "remains", "remained",
    })

    @staticmethod
    def _clean(span: Any) -> str:
        return span.strip() if isinstance(span, str) else ""

    @staticmethod
    def _normalize(span: str) -> str:
        return _normalize_identity(span)

    # ─────────────────────────────────────────────────────────────────────────
    # STEP 1 — Node extraction
    # ─────────────────────────────────────────────────────────────────────────

    def extract_nodes(self, annotation: Dict[str, Any]) -> Dict[str, List[Any]]:
        """
        Bucket the flat ``annotations`` list into typed groups.
        Now normalizes identity spans properly.
        """
        identities:  List[Dict[str, Any]] = []
        professions: List[str] = []
        triggers:    List[str] = []
        frames:      List[str] = []

        for ann in annotation.get("annotations", []):
            raw   = self._clean(ann.get("span", ""))
            if not raw:
                continue

            ntype = ann.get("node")

            # Only normalize identity nodes
            if ntype == self.IDENTITY_NODE:
                span = self._normalize(raw)
            else:
                span = raw.lower().strip()

            if ntype == self.IDENTITY_NODE:
                identities.append({
                    "span": span,
                    "label": ann.get("label"),
                    "category": _IDENTITY_TO_CATEGORY.get(span, "Other")
                })
            elif ntype == self.PROFESSION_NODE:
                professions.append(span)
            elif ntype == self.TRIGGER_NODE:
                triggers.append(span)
            elif ntype == self.FRAME_NODE:
                frames.append(span)

        return {
            "identities":  identities,
            "professions": professions,
            "triggers":    triggers,
            "frames":      frames,
        }

    # ─────────────────────────────────────────────────────────────────────────
    # STEP 2 — Frame classification
    # ─────────────────────────────────────────────────────────────────────────

    def _is_copula_frame(self, frame: str) -> bool:
        """
        True if frame is a copula/auxiliary verb (meaningless on its own).
        These become edge labels instead of nodes.
        """
        tokens = frame.strip().lower()
        return bool(tokens) and tokens in self.PREDICATE_LEAD_WORDS

    # ─────────────────────────────────────────────────────────────────────────
    # STEP 3 — Relation extraction (UPDATED)
    # ─────────────────────────────────────────────────────────────────────────

    def extract_relations(self, nodes):

      identities = nodes["identities"]
      professions = nodes["professions"]
      triggers = nodes["triggers"]
      frames = nodes["frames"]

      relations = []

      # ----------------------------
      # Identity ↔ Identity
      # ----------------------------
      if len(identities) > 1:
          relations.extend(
              self._build_intersections(identities)
          )

      # ----------------------------
      # Identity ↔ Profession
      # ----------------------------
      if professions:
          relations.extend(
              self._build_role_associations(
                  identities,
                  professions
              )
          )

      # ----------------------------
      # Frame-driven patterns
      # ----------------------------
      if len(frames) == 1 and len(triggers) > 1:
        frame_pairs = [(t, frames[0]) for t in triggers]

      else:
        frame_pairs = list(zip(triggers, frames))

      for trigger, frame in frame_pairs:

          if self._is_copula_frame(frame):

              relations.extend(
                  self._build_predicate_relations(
                      identities,
                      professions,
                      trigger,
                      frame
                  )
              )

          else:

              relations.extend(
                  self._build_semantic_relations(
                      identities,
                      professions,
                      trigger,
                      frame
                  )
              )

      return self._dedupe(relations)

    # ── Relation builders ─────────────────────────────────────────────────────

    def _build_intersections(
        self, identities: List[Dict[str, Any]]
    ) -> List[Dict[str, str]]:
        return [
            self._edge(a["span"], b["span"], self.REL_IDENTITY_INTERSECTION)
            for a, b in itertools.combinations(identities, 2)
        ]

    def _build_role_associations(
        self,
        identities: List[Dict[str, Any]],
        professions: List[str],
    ) -> List[Dict[str, str]]:
        return [
            self._edge(i["span"], p, self.REL_IDENTITY_ROLE_ASSOCIATION)
            for i in identities
            for p in professions
        ]

    def _build_predicate_relations(self,identities,professions,trigger,frame):

      relations = []

      for identity in identities:

          relations.append(
              self._edge(
                  identity["span"],
                  trigger,
                  frame
              )
          )

      for profession in professions:

          relations.append(
              self._edge(
                  profession,
                  trigger,
                  frame
              )
          )

      return relations

    def _build_semantic_relations(self,identities, professions,trigger,frame):

        relations = []

        # Identity → Frame

        for identity in identities:

            relations.append(
                self._edge(
                    identity["span"],
                    frame,
                    self.REL_IDENTITY_FRAME_ASSOCIATION
                )
            )

        # Profession → Frame

        for profession in professions:

            relations.append(
                self._edge(
                    profession,
                    frame,
                    self.REL_IDENTITY_ROLE_ASSOCIATION
                )
            )

        # Frame → Trigger

        relations.append(
            self._edge(
                frame,
                trigger,
                self.REL_CHARACTERIZED_AS
            )
        )

        return relations


    # ── Utilities ─────────────────────────────────────────────────────────────

    @staticmethod
    def _edge(source: str, target: str, relation: str) -> Dict[str, str]:
        return {"source": source, "target": target, "relation": relation}

    @staticmethod
    def _dedupe(rels: List[Dict[str, str]]) -> List[Dict[str, str]]:
        seen: set = set()
        unique: List[Dict[str, str]] = []
        for r in rels:
            k = (r["source"], r["target"], r["relation"])
            if k not in seen:
                seen.add(k)
                unique.append(r)
        return unique

    # ── Public entry points (unchanged) ───────────────────────────────────

    def process_annotation(self,annotation: Dict[str, Any]) -> List[Dict[str, str]]:
      nodes = self.extract_nodes(annotation)
      return self.extract_relations(nodes)

    def process_batch(self, annotations: List[Dict[str, Any]]) -> List[Dict[str, str]]:
        out: List[Dict[str, str]] = []
        for ann in annotations:
            out.extend(self.process_annotation(ann))
        return out

In [ ]:
# ── Per-sentence graph: typed nodes + safe handling of empty annotations ──────
# Each graph node is keyed by (normalised span, node type), so identical text
# assigned to different DER node types is preserved as distinct typed nodes.
# Empty annotations intentionally produce a sentence-only graph after add_sentence_node().

def _normalise_annotated_span(raw_span: str, node_type: str) -> str:
    raw = raw_span.strip()
    return _normalize_identity(raw) if node_type == "Identity Dimensions" else raw.lower().strip()


def _typed_node_id(span: str, node_type: str):
    return (span, node_type)


def _lookup_node_meta(span: str, raw_annotations: list) -> Tuple[str, Optional[str]]:
    """Compatibility helper: return the first matching annotation's type and label."""
    for ann in raw_annotations:
        raw = ann.get("span", "")
        if not isinstance(raw, str):
            continue
        node_type = ann.get("node", "Unknown")
        if _normalise_annotated_span(raw, node_type) == span:
            return node_type, ann.get("label")
    return "Unknown", None


def build_sentence_graph(
    annotation: Dict[str, Any],
    extractor: Optional[RuleBasedRelations] = None
) -> nx.DiGraph:
    if extractor is None:
        extractor = RuleBasedRelations()

    G = nx.DiGraph()
    raw_anns = annotation.get("annotations", []) or []

    # span -> all typed node IDs for that normalised text; keep duplicates by type.
    candidates = defaultdict(list)

    # 1. Add every valid annotation, including isolated nodes.
    for ann in raw_anns:
        raw_span = ann.get("span", "")
        if not isinstance(raw_span, str) or not raw_span.strip():
            continue
        node_type = ann.get("node", "Unknown")
        span = _normalise_annotated_span(raw_span, node_type)
        if not span:
            continue
        node_id = _typed_node_id(span, node_type)
        if node_id not in G:
            G.add_node(
                node_id,
                node_type=node_type,
                ontology_label=ann.get("label"),
                frequency=1,
                surface=raw_span.strip(),
            )
            candidates[span].append(node_id)
        else:
            G.nodes[node_id]["frequency"] += 1

    # Type expectations disambiguate relation endpoints that the current rule
    # extractor emits as plain text strings rather than typed node references.
    ID = RuleBasedRelations.IDENTITY_NODE
    PROF = RuleBasedRelations.PROFESSION_NODE
    TRIG = RuleBasedRelations.TRIGGER_NODE
    FRAME = RuleBasedRelations.FRAME_NODE
    expected_types = {
        RuleBasedRelations.REL_IDENTITY_INTERSECTION: ({ID}, {ID}),
        RuleBasedRelations.REL_CHARACTERIZED_AS: ({FRAME}, {TRIG}),
        RuleBasedRelations.REL_IDENTITY_FRAME_ASSOCIATION: ({ID}, {FRAME}),
        RuleBasedRelations.REL_IDENTITY_ROLE_ASSOCIATION: ({ID, PROF}, {PROF, FRAME}),
    }

    def resolve_endpoint(text, preferred_types=None):
        text = str(text).strip()
        matches = candidates.get(text, [])
        if preferred_types:
            preferred = [n for n in matches if G.nodes[n].get("node_type") in preferred_types]
            if preferred:
                matches = preferred
        if matches:
            return matches[0]
        # Preserve any unexpected relation endpoint as an Unknown typed node.
        node_id = _typed_node_id(text, "Unknown")
        if node_id not in G:
            G.add_node(node_id, node_type="Unknown", ontology_label=None,
                       frequency=1, surface=text)
        candidates[text].append(node_id)
        return node_id

    # 2. Add extracted relations, mapped onto type-aware node IDs.
    for rel in extractor.process_annotation(annotation):
        source_text, target_text = rel["source"], rel["target"]
        relation = rel["relation"]
        src_types, tgt_types = expected_types.get(relation, ({ID, PROF}, {TRIG}))
        source = resolve_endpoint(source_text, src_types)
        target = resolve_endpoint(target_text, tgt_types)
        if G.has_edge(source, target):
            G[source][target]["weight"] += 1
        else:
            G.add_edge(source, target, relation=relation, weight=1)

    G.graph["sentence"] = annotation.get("sentence", "")
    return G


In [ ]:
SENTENCE_NODE_TYPE = "Sentence"
SENTENCE_RELATION = "contains"


def add_sentence_node(G):
    """
    Add a dedicated Sentence node to an existing
    sentence-level NetworkX graph.
    """

    G = G.copy()

    sentence = G.graph.get("sentence", "")

    # Unique node name for this graph
    sentence_node = "__SENTENCE__"

    # Add sentence node
    G.add_node(
        sentence_node,
        node_type=SENTENCE_NODE_TYPE,
        ontology_label=None,
        frequency=1,
        text=sentence
    )

    # Connect Sentence -> every existing semantic node
    for node in list(G.nodes()):

        if node == sentence_node:
            continue

        G.add_edge(
            sentence_node,
            node,
            relation=SENTENCE_RELATION,
            weight=1
        )

    return G

In [ ]:
# Remember each graph node's original surface span (needed for contextual BERT alignment).
IDENTITY_NODE_NAME = "Identity Dimensions"

def _raw_span_for_graph_node(node, node_attrs, annotation):
    """Return the stored original span; fall back to annotation matching for older graphs."""
    if node_attrs.get("surface"):
        return str(node_attrs["surface"])
    ntype = node_attrs.get("node_type", "Unknown")
    node_text = node[0] if isinstance(node, tuple) and len(node) == 2 else str(node)
    for ann in annotation.get("annotations", []):
        if ann.get("node", "Unknown") != ntype:
            continue
        raw = str(ann.get("span", "")).strip()
        if not raw:
            continue
        cand = _normalize_identity(raw) if ntype == IDENTITY_NODE_NAME else raw.lower().strip()
        if cand == str(node_text).lower().strip():
            return raw
    return str(node_text)


def attach_surfaces(G, annotation):
    for n, a in G.nodes(data=True):
        if a.get("node_type") == SENTENCE_NODE_TYPE:
            continue
        a["surface"] = _raw_span_for_graph_node(n, a, annotation)
    return G


In [ ]:
IDENTITY_NODE = RuleBasedRelations.IDENTITY_NODE
CANONICAL_RELATIONS = {
    RuleBasedRelations.REL_IDENTITY_INTERSECTION,
    RuleBasedRelations.REL_IDENTITY_ROLE_ASSOCIATION,
    RuleBasedRelations.REL_CHARACTERIZED_AS,
    RuleBasedRelations.REL_IDENTITY_FRAME_ASSOCIATION,
    SENTENCE_RELATION,
}


def relation_family(rel: str) -> str:
    """Collapse the open-ended relation vocabulary (copula words such as 'is', 'can', ...)
    into 'predicate'; canonical DER relations and 'contains' are kept. Strips 'rev_'."""
    r = rel[4:] if rel.startswith("rev_") else rel
    return r if r in CANONICAL_RELATIONS else "predicate"


# ── NetworkX-level graph edits (used by ablations / counterfactuals / occlusion) ──
def nx_remove_der_edges(G, families=None):
    """Remove DER edges (everything except Sentence->node 'contains').
    families=None removes all; otherwise only edges whose relation_family is in `families`."""
    G2 = G.copy()
    drop = [(u, v) for u, v, a in G2.edges(data=True)
            if a["relation"] != SENTENCE_RELATION
            and (families is None or relation_family(a["relation"]) in families)]
    G2.remove_edges_from(drop)
    return G2


def nx_remove_node_type(G, node_type):
    G2 = G.copy()
    G2.remove_nodes_from([n for n, a in G2.nodes(data=True) if a.get("node_type") == node_type])
    return G2


# ── Build graphs for each dataset ───────────────────────────────────────────
extractor = RuleBasedRelations()


def build_dataset(annos_pos, annos_neg):
    annos = list(annos_pos) + list(annos_neg)
    labels = np.array([1] * len(annos_pos) + [0] * len(annos_neg))
    graphs = [attach_surfaces(add_sentence_node(build_sentence_graph(a, extractor)), a) for a in annos]
    # Groups: same template_id => same group (pair-safe CV). Items without one get their own group.
    groups = np.array([a["template_id"] if a.get("template_id") is not None else -(i + 1)
                       for i, a in enumerate(annos)])
    return dict(annos=annos, graphs=graphs, labels=labels, groups=groups)


DATA = {name: build_dataset(d["pos"], d["neg"]) for name, d in DATASETS.items()}
for name, d in DATA.items():
    n_nodes = np.mean([G.number_of_nodes() - 1 for G in d["graphs"]])
    n_der = np.mean([sum(a["relation"] != SENTENCE_RELATION for _, _, a in G.edges(data=True))
                     for G in d["graphs"]])
    print(f"{name}: {len(d['graphs'])} graphs | avg entity nodes {n_nodes:.1f} | avg DER edges {n_der:.1f}"
          f" | groups {len(set(d['groups']))}")

InDiBias: 326 graphs | avg entity nodes 2.8 | avg DER edges 1.6 | groups 326
IndiCasa: 326 graphs | avg entity nodes 5.8 | avg DER edges 4.7 | groups 326


In [ ]:
# ── Graph coverage audit (run after DATA is built) ────────────────────────────
# Empty annotations are expected to have zero DER/entity nodes; they are not
# silently repaired or assigned invented relationships.
for name, d in DATA.items():
    rows = []
    for i, (ann, G, y) in enumerate(zip(d["annos"], d["graphs"], d["labels"])):
        entity_nodes = [(n, a) for n, a in G.nodes(data=True)
                        if a.get("node_type") != SENTENCE_NODE_TYPE]
        der_edges = [(u, v, a) for u, v, a in G.edges(data=True)
                     if a.get("relation") != SENTENCE_RELATION]
        if not ann.get("annotations") or not entity_nodes:
            rows.append({
                "dataset": name, "index": i, "label": int(y),
                "sentence": ann.get("sentence", ""),
                "annotation_count": len(ann.get("annotations", []) or []),
                "entity_node_count": len(entity_nodes),
                "der_edge_count": len(der_edges),
                "empty_annotations": not bool(ann.get("annotations")),
            })
    print(f"{name}: {len(rows)} records with empty annotations or no entity nodes")
    if rows:
        display(pd.DataFrame(rows).head(10))


InDiBias: 40 records with empty annotations or no entity nodes


,dataset,index,label,sentence,annotation_count,entity_node_count,der_edge_count,empty_annotations
0,InDiBias,27,1,They were asked to be careful while walking th...,0,0,0,True
1,InDiBias,73,1,Prakash learned to play chess very quickly and...,0,0,0,True
2,InDiBias,74,1,Anjali was terrified of bugs.,0,0,0,True
3,InDiBias,78,1,Kabir will not let his sister go for buying mi...,0,0,0,True
4,InDiBias,81,1,Saira served food in a restaurant.,0,0,0,True
5,InDiBias,86,1,Imran was the fastest on the draw and shot roh...,0,0,0,True
6,InDiBias,87,1,Raj took off his ring and put it in his pocket...,0,0,0,True
7,InDiBias,88,1,Raj took his wedding ring off so it wouldn't g...,0,0,0,True
8,InDiBias,89,1,Sunil spent most of his time writing programs ...,0,0,0,True
9,InDiBias,96,1,i'm glad i wasn't there since i would have bee...,0,0,0,True


IndiCasa: 1 records with empty annotations or no entity nodes


,dataset,index,label,sentence,annotation_count,entity_node_count,der_edge_count,empty_annotations
0,IndiCasa,108,1,The brother will not let his sister go for buy...,0,0,0,True


## 3. Frozen text encoders and graph conversion

The encoder weights are frozen. We cache their node vectors once, then train only the RGAT classifier. The graph model receives DER nodes and DER-to-DER relations, not the sentence node.


In [ ]:
# ── graph-attribute helpers ────────────────────────────────────────────────
def sentence_node_name(G):
    return next(n for n, a in G.nodes(data=True) if a.get("node_type") == SENTENCE_NODE_TYPE)


def sentence_text(G):
    a = G.nodes[sentence_node_name(G)]
    return a.get("text", G.graph.get("sentence", ""))


def node_surface(G, n):          # original span text (falls back to the node name)
    return G.nodes[n].get("surface", str(n))


def node_ctx(G, n):              # sentence the node is contextualised in (counterfactual node-only swaps override this)
    return G.nodes[n].get("ctx", G.graph.get("sentence", ""))


def find_char_span(sentence, span_text):
    """Case-insensitive, word-bounded span lookup (handles simple plurals)."""
    if not sentence or not span_text:
        return None
    cands = [span_text]
    if span_text.lower().endswith("y"):
        cands.append(span_text[:-1] + "ies")
    cands += [span_text + "s", span_text + "es"]
    for c in cands:
        m = re.search(r"(?<!\w)" + re.escape(c) + r"(?!\w)", sentence, flags=re.IGNORECASE)
        if m:
            return m.start(), m.end()
    m = re.search(re.escape(span_text), sentence, flags=re.IGNORECASE)   # last resort
    return (m.start(), m.end()) if m else None


# ── featurisers ────────────────────────────────────────────────────────────
class Featurizer:
    """text/span -> vector cache. keys are hashable; vectors live on CPU in fp32."""
    name, dim = "?", 0

    def __init__(self):
        self.cache, self.failures, self.released = {}, set(), False

    def node_key(self, G, n):
        raise NotImplementedError

    def _compute(self, missing):
        raise NotImplementedError

    def keys(self, G, nodes=None):
        nodes = list(G.nodes) if nodes is None else nodes
        return [self.node_key(G, n) for n in nodes]

    def ensure(self, keys):
        missing = list(dict.fromkeys(k for k in keys if k not in self.cache))
        if not missing:
            return
        if self.released:
            raise RuntimeError(f"{self.name} encoder was released but {len(missing)} uncached keys were requested "
                               f"(e.g. {missing[0]}). Add those graphs to the precompute step in section 5.")
        self._compute(missing)

    def warm(self, graphs):
        self.ensure([k for G in graphs for k in self.keys(G)])

    def vectors(self, G, nodes):
        ks = self.keys(G, nodes)
        self.ensure(ks)
        return torch.stack([self.cache[k] for k in ks])

    def sentence_vector(self, G):
        k = self.node_key(G, sentence_node_name(G))
        self.ensure([k])
        return self.cache[k]


class SBERTFeaturizer(Featurizer):
    name = "SBERT"

    def __init__(self, model_name="all-MiniLM-L6-v2"):
        super().__init__()
        from sentence_transformers import SentenceTransformer
        self.model = SentenceTransformer(model_name, device=str(device))
        if ENCODER_FP16 and device.type == "cuda":
            self.model.half()
        self.dim = int(self.model.get_sentence_embedding_dimension())
        self.model_name = model_name

    def node_key(self, G, n):
        a = G.nodes[n]
        if a.get("node_type") == SENTENCE_NODE_TYPE:
            return ("t", a.get("text", G.graph.get("sentence", "")))
        return ("t", node_surface(G, n))          # DER span text only; no full-sentence or typed-ID leakage

    def _compute(self, missing):
        texts = [k[1] for k in missing]
        with torch.no_grad():
            embs = self.model.encode(texts, batch_size=ENC_BATCH, convert_to_tensor=True, show_progress_bar=False)
        embs = embs.float().cpu()
        for k, e in zip(missing, embs):
            self.cache[k] = e.clone()

    def release(self):
        self.model = None; self.released = True; free_gpu()


class BERTFeaturizer(Featurizer):
    name = "BERT"

    def __init__(self, model_name="bert-base-uncased"):
        super().__init__()
        from transformers import AutoTokenizer, AutoModel
        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.model = AutoModel.from_pretrained(model_name).to(device).eval()
        for p in self.model.parameters():
            p.requires_grad = False
        self.dim = int(self.model.config.hidden_size)
        self.model_name = model_name
        self.fp16 = bool(ENCODER_FP16 and device.type == "cuda")

    def node_key(self, G, n):
        a = G.nodes[n]
        if a.get("node_type") == SENTENCE_NODE_TYPE:
            return ("cls", a.get("text", G.graph.get("sentence", "")))
        return ("span", node_ctx(G, n), node_surface(G, n))

    def _compute(self, missing):
        by_sent = defaultdict(list)
        for k in missing:
            by_sent[k[1]].append(k)
        sents = sorted(by_sent, key=len)          # length-sorted batches => little padding
        for b in range(0, len(sents), ENC_BATCH):
            chunk = sents[b:b + ENC_BATCH]
            enc = self.tok(chunk, return_tensors="pt", padding=True, truncation=True,
                           max_length=256, return_offsets_mapping=True)
            offsets = enc.pop("offset_mapping").numpy()
            enc = {k: v.to(device) for k, v in enc.items()}
            with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=self.fp16):
                hidden = self.model(**enc).last_hidden_state
            hidden = hidden.float().cpu()          # [B, T, 768] -> CPU immediately
            for i, s in enumerate(chunk):
                for k in by_sent[s]:
                    if k[0] == "cls":
                        vec = hidden[i, 0]
                    else:
                        span = find_char_span(s, k[2])
                        ids = [] if span is None else [j for j, (a, c) in enumerate(offsets[i])
                                                       if c > a and a < span[1] and c > span[0]]
                        if ids:
                            vec = hidden[i, ids].mean(dim=0)
                        else:
                            vec = torch.zeros(self.dim); self.failures.add((s, k[2]))
                    self.cache[k] = vec.clone()
            del hidden, enc

    def release(self):
        self.model = None; self.tok = None; self.released = True; free_gpu()


class BERTIsolatedFeaturizer(BERTFeaturizer):
    """BERT CLS vectors for each DER node's text in isolation; sentence nodes stay contextual CLS."""
    name = "BERT-ISO"

    def node_key(self, G, n):
        a = G.nodes[n]
        if a.get("node_type") == SENTENCE_NODE_TYPE:
            return ("cls", a.get("text", G.graph.get("sentence", "")))
        return ("isolated", node_surface(G, n))

    def _compute(self, missing):
        # Sentence vectors use full-sentence CLS. DER nodes use standalone node text CLS.
        sentence_keys = [k for k in missing if k[0] == "cls"]
        isolated_keys = [k for k in missing if k[0] == "isolated"]
        batches = [(sentence_keys, [k[1] for k in sentence_keys]),
                   (isolated_keys, [k[1] for k in isolated_keys])]
        for keys, texts in batches:
            if not keys:
                continue
            for start in range(0, len(keys), ENC_BATCH):
                key_chunk = keys[start:start + ENC_BATCH]
                text_chunk = texts[start:start + ENC_BATCH]
                enc = self.tok(text_chunk, return_tensors="pt", padding=True,
                               truncation=True, max_length=256)
                enc = {k: v.to(device) for k, v in enc.items()}
                with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=self.fp16):
                    hidden = self.model(**enc).last_hidden_state[:, 0, :]
                hidden = hidden.float().cpu()
                for k, vec in zip(key_chunk, hidden):
                    self.cache[k] = vec.clone()
                del hidden, enc


FEATURIZERS: Dict[str, Featurizer] = {}


In [ ]:
def nx_to_hetero(G, label, feat, collapse_relations=False, zero_sentence=False):
    """Convert a NetworkX sentence graph to a DER-only HeteroData graph.

    The Sentence node is deliberately excluded. Only annotated DER nodes,
    their text-only features, and relations between DER nodes reach the R-GCN.
    Edges touching the Sentence node are discarded.
    """
    data = HeteroData()
    node_maps = {}
    der_nodes = [(n, a) for n, a in G.nodes(data=True)
                 if a.get("node_type", "Unknown") != SENTENCE_NODE_TYPE]
    node_types = sorted({a.get("node_type", "Unknown") for _, a in der_nodes})

    for nt in node_types:
        nodes = [n for n, a in der_nodes if a.get("node_type", "Unknown") == nt]
        node_maps[nt] = {n: i for i, n in enumerate(nodes)}
        feat.ensure([feat.node_key(G, n) for n in nodes])
        data[nt].x = torch.stack([feat.get(G, n) for n in nodes]).float()

    edge_dict = defaultdict(list)
    for s, t, a in G.edges(data=True):
        st = G.nodes[s].get("node_type", "Unknown")
        tt = G.nodes[t].get("node_type", "Unknown")
        # No Sentence node or Sentence-incident edges are allowed into the encoder.
        if st == SENTENCE_NODE_TYPE or tt == SENTENCE_NODE_TYPE:
            continue
        if st not in node_maps or tt not in node_maps:
            continue
        rel = a.get("relation", "unknown")
        if collapse_relations:
            rel = relation_family(rel)
        si, ti = node_maps[st][s], node_maps[tt][t]
        edge_dict[(st, rel, tt)].append([si, ti])
        edge_dict[(tt, "rev_" + rel, st)].append([ti, si])

    for et, e in edge_dict.items():
        data[et].edge_index = torch.tensor(e, dtype=torch.long).t().contiguous()
    data.y = torch.tensor([int(label)], dtype=torch.long)
    return data


def convert(graphs_nx, labels, feat, **kw):
    return [nx_to_hetero(G, y, feat, **kw) for G, y in zip(graphs_nx, labels)]


def build_metadata(data_list):
    nts, ets = set(), set()
    for g in data_list:
        n, e = g.metadata()
        nts.update(n); ets.update(e)
    return sorted(nts), sorted(ets)


In [ ]:
# Cache DER-node vectors for SBERT and BERT-iso. Sentence-node vectors are not used.
FEATURIZERS = {}
for _fname in ["SBERT", "BERT-ISO"]:
    _t0 = time.time()
    _feat = SBERTFeaturizer() if _fname == "SBERT" else BERTIsolatedFeaturizer()
    _der_keys = []
    for _d in DATA.values():
        for _G in _d["graphs"]:
            for _n, _a in _G.nodes(data=True):
                if _a.get("node_type") != SENTENCE_NODE_TYPE:
                    _der_keys.append(_feat.node_key(_G, _n))
    _feat.ensure(_der_keys)
    _nvec = len(_feat.cache)
    _feat.release()
    FEATURIZERS[_fname] = _feat
    print(f"{_fname}: cached {_nvec} DER-node vectors in {time.time()-_t0:.1f}s")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT-ISO: cached 933 DER-node vectors in 26.7s


In [ ]:
from transformers import AutoTokenizer, AutoModel

class BERTContextFeaturizer:
    """Mean token embedding for each annotated span in its full sentence context."""
    dim = 768
    def __init__(self):
        self.tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased", use_fast=True)
        self.model = AutoModel.from_pretrained("bert-base-uncased").to(device).eval()
        for p in self.model.parameters(): p.requires_grad = False
        self.cache = {}
    def node_key(self, G, node):
        attrs = G.nodes[node]
        return (G.graph.get("sentence", ""), attrs.get("node_type", "Unknown"), attrs.get("surface", str(node)))
    @torch.no_grad()
    def _encode(self, sentence, spans):
        enc = self.tokenizer(sentence, return_tensors="pt", return_offsets_mapping=True,
                             truncation=True, max_length=256)
        offsets = enc.pop("offset_mapping")[0].tolist()
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.autocast(device_type="cuda", dtype=torch.float16,
                            enabled=(ENCODER_FP16 and device.type == "cuda")):
            hidden = self.model(**enc).last_hidden_state[0].float().cpu()
        out = {}
        for span in spans:
            start = sentence.lower().find(span.lower())
            if start < 0:
                out[span] = np.zeros(self.dim, dtype=np.float32); continue
            end = start + len(span)
            ids = [i for i, (a, b) in enumerate(offsets) if b > a and a < end and b > start]
            out[span] = (hidden[ids].mean(0).numpy().astype(np.float32) if ids
                         else np.zeros(self.dim, dtype=np.float32))
        return out
    def ensure(self, keys):
        by_sentence = {}
        for sentence, node_type, span in keys:
            if (sentence, node_type, span) not in self.cache:
                by_sentence.setdefault(sentence, set()).add(span)
        for sentence, spans in by_sentence.items():
            encoded = self._encode(sentence, sorted(spans))
            for span in spans:
                self.cache[(sentence, None, span)] = encoded[span]
        for sentence, node_type, span in keys:
            self.cache[(sentence, node_type, span)] = self.cache[(sentence, None, span)]
    def get(self, G, node):
        return torch.from_numpy(self.cache[self.node_key(G, node)])
    def release(self):
        self.model.to("cpu"); del self.model
        if torch.cuda.is_available(): torch.cuda.empty_cache()

_t0 = time.time()
bert_ctx_feat = BERTContextFeaturizer()
ctx_keys = []
for _d in DATA.values():
    for _G in _d["graphs"]:
        for _node, _attrs in _G.nodes(data=True):
            if _attrs.get("node_type") != SENTENCE_NODE_TYPE:
                ctx_keys.append(bert_ctx_feat.node_key(_G, _node))
bert_ctx_feat.ensure(ctx_keys)
ctx_nvec = len(bert_ctx_feat.cache)
bert_ctx_feat.release()
FEATURIZERS["BERT-CTX"] = bert_ctx_feat
print(f"BERT-ctx: cached {ctx_nvec} DER-node vectors in {time.time()-_t0:.1f}s")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT-ctx: cached 5097 DER-node vectors in 69.3s


## 4. Convert typed DER graphs for RGAT

RGAT operates on a homogeneous edge-index representation with an explicit `edge_type` ID. We preserve node-type IDs and use separate input projections for each DER node type. Both forward and reverse directed edges are kept, matching the existing graph conversion.


In [ ]:
def get_hetero_graphs_for_mode(dataset_name, encoder_name, relation_mode):
    feat = FEATURIZERS[encoder_name]
    if relation_mode not in {"raw", "collapsed"}:
        raise ValueError(f"Unknown relation mode: {relation_mode}")
    return convert(DATA[dataset_name]["graphs"], DATA[dataset_name]["labels"], feat,
                   collapse_relations=(relation_mode == "collapsed"))


def build_metadata(data_list):
    node_types, edge_types = set(), set()
    for g in data_list:
        n, e = g.metadata()
        node_types.update(n); edge_types.update(e)
    return sorted(node_types), sorted(edge_types)


def to_homogeneous_graphs(hetero_graphs, node_types, edge_types, input_dim):
    """Flatten each HeteroData graph while retaining global node/relation type IDs."""
    nt2id = {nt: i for i, nt in enumerate(node_types)}
    et2id = {et: i for i, et in enumerate(edge_types)}
    out = []
    for hg in hetero_graphs:
        xs, node_type_ids, offsets = [], [], {}
        offset = 0

        for nt in node_types:
            if nt in hg.node_types and hasattr(hg[nt], "x"):
                x_nt = hg[nt].x.float()
            else:
                x_nt = torch.empty((0, input_dim), dtype=torch.float32)
            offsets[nt] = offset
            offset += x_nt.size(0)
            xs.append(x_nt)
            node_type_ids.extend([nt2id[nt]] * x_nt.size(0))
        x = torch.cat(xs, dim=0) if xs else torch.empty((0, input_dim), dtype=torch.float32)
        edge_parts, relation_ids = [], []
        for et in edge_types:
            if et not in hg.edge_types:
                continue
            ei = hg[et].edge_index
            if ei.numel() == 0:
                continue
            src_type, _, dst_type = et
            ei = ei.clone()
            ei[0] += offsets[src_type]
            ei[1] += offsets[dst_type]
            edge_parts.append(ei)
            relation_ids.extend([et2id[et]] * ei.size(1))
        edge_index = torch.cat(edge_parts, dim=1) if edge_parts else torch.empty((2, 0), dtype=torch.long)
        out.append(Data(
            x=x,
            node_type=torch.tensor(node_type_ids, dtype=torch.long),
            edge_index=edge_index,
            edge_type=torch.tensor(relation_ids, dtype=torch.long),
            y=hg.y.clone().long(),
        ))
    return out


In [ ]:
class RGATGraphClassifier(nn.Module):
    """Two relation-aware attention layers followed by mean graph pooling."""
    def __init__(self, node_types, num_relations, input_dim, hidden=64, dropout=0.3, heads=2):
        super().__init__()
        self.node_types = list(node_types)
        self.dropout = dropout
        self.input_proj = nn.ModuleList([nn.Linear(input_dim, hidden) for _ in self.node_types])
        self.rgat1 = RGATConv(hidden, hidden, num_relations=max(1, num_relations),
                              heads=heads, concat=False, dropout=dropout)
        self.rgat2 = RGATConv(hidden, hidden, num_relations=max(1, num_relations),
                              heads=heads, concat=False, dropout=dropout)
        self.classifier = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), nn.Dropout(dropout),
                                        nn.Linear(32, 2))

    def forward(self, data):
        n_graphs = int(data.num_graphs)
        x = data.x
        if x.size(0) == 0:
            pooled = x.new_zeros((n_graphs, self.classifier[0].in_features))
            return self.classifier(pooled)

        # Node-type-specific input projections.
        projected = x.new_zeros((x.size(0), self.input_proj[0].out_features))
        for type_id, layer in enumerate(self.input_proj):
            mask = data.node_type == type_id
            if mask.any():
                projected[mask] = layer(x[mask])
        x = F.relu(projected)

        # Some graphs have no DER edges. Skip message passing for an entirely edgeless batch.
        if data.edge_index.numel() > 0:
            x = F.relu(self.rgat1(x, data.edge_index, data.edge_type))
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = F.relu(self.rgat2(x, data.edge_index, data.edge_type))

        pooled = global_mean_pool(x, data.batch, size=n_graphs)
        return self.classifier(pooled)


## 5. Evaluation: accuracy and F1 only

The notebook uses the same split strategy across all three encoders. `full` runs three repeats of five-fold evaluation for the three plans. Use `quick` first to verify that dependencies and the four data files load correctly.


In [ ]:
def make_splits(labels, groups, strategy, seeds, n_splits=5):
    reps = []
    for seed in seeds:
        if strategy == "stratified":
            splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
            folds = splitter.split(np.zeros(len(labels)), labels)
        elif strategy == "grouped":
            splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
            folds = splitter.split(np.zeros(len(labels)), labels, groups)
        else:
            raise ValueError(f"Unknown split strategy: {strategy}")
        reps.append([(tr, te) for tr, te in folds])
    return reps


def fit_rgat(train_graphs, node_types, edge_types, input_dim, seed, config_name):
    config = CONFIGS[config_name]
    set_seed(seed)
    model = RGATGraphClassifier(
        node_types=node_types,
        num_relations=len(edge_types),
        input_dim=input_dim,
        hidden=HIDDEN,
        dropout=config["dropout"],
        heads=RGAT_HEADS,
    ).to(train_device)
    loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=LR, weight_decay=config["weight_decay"]
    )
    for _epoch in range(EPOCHS):
        model.train()
        for batch in loader:
            batch = batch.to(train_device)
            optimizer.zero_grad()
            logits = model(batch)
            loss = F.cross_entropy(logits, batch.y.view(-1))
            loss.backward()
            optimizer.step()
    return model


@torch.no_grad()
def predict_labels(model, graphs, batch_size=64):
    model.eval()
    predictions = []
    for batch in DataLoader(graphs, batch_size=batch_size, shuffle=False):
        logits = model(batch.to(next(model.parameters()).device))
        predictions.extend(logits.argmax(dim=1).cpu().numpy().tolist())
    return np.asarray(predictions, dtype=int)


def evaluate_repeated_cv(graphs, labels, groups, strategy, encoder_name, dataset_name, config_name):
    split_repeats = make_splits(labels, groups, strategy, SEEDS)
    per_repeat = []
    for rep_idx, (folds, seed) in enumerate(zip(split_repeats, SEEDS), start=1):
        oof = np.full(len(labels), -1, dtype=int)
        for fold_idx, (train_idx, test_idx) in enumerate(folds, start=1):
            model = fit_rgat(
                [graphs[i] for i in train_idx], NODE_TYPES, EDGE_TYPES,
                INPUT_DIM, seed + fold_idx, config_name
            )
            oof[test_idx] = predict_labels(model, [graphs[i] for i in test_idx])
            del model
            free_gpu()
        if np.any(oof < 0):
            raise RuntimeError("Some examples did not receive an out-of-fold prediction")
        per_repeat.append({
            "dataset": dataset_name, "cv": strategy, "encoder": encoder_name,
            "config": config_name, "repeat": rep_idx,
            "accuracy": accuracy_score(labels, oof),
            "macro_f1": f1_score(labels, oof, average="macro", zero_division=0),
            "weighted_f1": f1_score(labels, oof, average="weighted", zero_division=0),
        })
        print(f"  {config_name} | repeat {rep_idx}/{N_REPEATS}: "
              f"accuracy={per_repeat[-1]['accuracy']:.4f}, "
              f"macro-F1={per_repeat[-1]['macro_f1']:.4f}, "
              f"weighted-F1={per_repeat[-1]['weighted_f1']:.4f}")
    return per_repeat


In [ ]:
# Dynamically add the missing helper method .get(G, n) to the Featurizer classes to fix the AttributeError
for name, feat in FEATURIZERS.items():
    if not hasattr(feat, 'get'):
        def make_get_fn(f):
            return lambda G, n: f.cache[f.node_key(G, n)]
        feat.get = make_get_fn(feat)

ENCODERS = ["SBERT", "BERT-ISO", "BERT-CTX"]
all_repeat_rows = []
summary_rows = []
t_start = time.time()

for dataset_name, cv_strategy in EVAL_PLAN:
    labels = DATA[dataset_name]["labels"]
    groups = DATA[dataset_name]["groups"]
    print(f"\\n=== {dataset_name} | {cv_strategy} CV ===")
    for relation_mode in RELATION_MODES:
        print(f"\\n--- Relation mode: {relation_mode} ---")
        for encoder_name in ENCODERS:
            t0 = time.time()
            # Build graph representations separately for each relation mode.
            hetero_graphs = get_hetero_graphs_for_mode(dataset_name, encoder_name, relation_mode)
            NODE_TYPES, EDGE_TYPES = build_metadata(hetero_graphs)
            INPUT_DIM = FEATURIZERS[encoder_name].dim
            homogeneous_graphs = to_homogeneous_graphs(
                hetero_graphs, NODE_TYPES, EDGE_TYPES, INPUT_DIM
            )
            if any("Sentence" in nt for nt in NODE_TYPES):
                raise AssertionError("Sentence nodes must not be passed to the DER-only RGAT")
            print(f"Encoder={encoder_name} | node types={len(NODE_TYPES)} | "
                  f"relation types={len(EDGE_TYPES)} | graphs={len(homogeneous_graphs)}")

            for config_name in CONFIGS:
                rows = evaluate_repeated_cv(
                    homogeneous_graphs, labels, groups, cv_strategy,
                    encoder_name, dataset_name, config_name
                )
                for row in rows:
                    row["relation_mode"] = relation_mode
                all_repeat_rows.extend(rows)
                frame = pd.DataFrame(rows)
                summary_rows.append({
                    "dataset": dataset_name,
                    "cv": cv_strategy,
                    "encoder": encoder_name,
                    "config": config_name,
                    "relation_mode": relation_mode,
                    "accuracy_mean": frame["accuracy"].mean(),
                    "accuracy_sd": frame["accuracy"].std(ddof=0),
                    "macro_f1_mean": frame["macro_f1"].mean(),
                    "macro_f1_sd": frame["macro_f1"].std(ddof=0),
                    "weighted_f1_mean": frame["weighted_f1"].mean(),
                    "weighted_f1_sd": frame["weighted_f1"].std(ddof=0),
                    "n_repeats": N_REPEATS,
                })
                latest = summary_rows[-1]
                print(f"SUMMARY {encoder_name} {config_name} {relation_mode}: "
                      f"acc={latest['accuracy_mean']:.4f}±{latest['accuracy_sd']:.4f}; "
                      f"macro-F1={latest['macro_f1_mean']:.4f}±{latest['macro_f1_sd']:.4f}; "
                      f"time={time.time()-t0:.1f}s")

            del homogeneous_graphs, hetero_graphs
            free_gpu()

repeat_results_df = pd.DataFrame(all_repeat_rows)
results_df = pd.DataFrame(summary_rows)
repeat_results_df.to_csv("results/rgat_repeat_results_base_reg_raw_collapsed.csv", index=False)
results_df.to_csv("results/rgat_accuracy_f1_results_base_reg_raw_collapsed.csv", index=False)
print(f"\\nCompleted in {(time.time()-t_start)/60:.1f} minutes")
display(results_df.round(4))
print("Saved results/rgat_repeat_results_base_reg_raw_collapsed.csv and "
      "results/rgat_accuracy_f1_results_base_reg_raw_collapsed.csv")


\n=== InDiBias | grouped CV ===
\n--- Relation mode: collapsed ---
Encoder=SBERT | node types=4 | relation types=10 | graphs=326
  base | repeat 1/3: accuracy=0.9264, macro-F1=0.9052, weighted-F1=0.9272
  base | repeat 2/3: accuracy=0.9571, macro-F1=0.9420, weighted-F1=0.9565
  base | repeat 3/3: accuracy=0.9387, macro-F1=0.9198, weighted-F1=0.9389
SUMMARY SBERT base collapsed: acc=0.9407±0.0126; macro-F1=0.9224±0.0151; time=129.7s
  reg | repeat 1/3: accuracy=0.9356, macro-F1=0.9148, weighted-F1=0.9355
  reg | repeat 2/3: accuracy=0.9387, macro-F1=0.9198, weighted-F1=0.9389
  reg | repeat 3/3: accuracy=0.9387, macro-F1=0.9198, weighted-F1=0.9389
SUMMARY SBERT reg collapsed: acc=0.9376±0.0014; macro-F1=0.9181±0.0024; time=261.0s
Encoder=BERT-ISO | node types=4 | relation types=10 | graphs=326
  base | repeat 1/3: accuracy=0.9356, macro-F1=0.9141, weighted-F1=0.9352
  base | repeat 2/3: accuracy=0.9417, macro-F1=0.9223, weighted-F1=0.9414
  base | repeat 3/3: accuracy=0.9294, macro-F1=0

,dataset,cv,encoder,config,relation_mode,accuracy_mean,accuracy_sd,macro_f1_mean,macro_f1_sd,weighted_f1_mean,weighted_f1_sd,n_repeats
0,InDiBias,grouped,SBERT,base,collapsed,0.9407,0.0126,0.9224,0.0151,0.9409,0.0120,3
1,InDiBias,grouped,SBERT,reg,collapsed,0.9376,0.0014,0.9181,0.0024,0.9377,0.0016,3
2,InDiBias,grouped,BERT-ISO,base,collapsed,0.9356,0.0050,0.9144,0.0064,0.9353,0.0049,3
3,InDiBias,grouped,BERT-ISO,reg,collapsed,0.9376,0.0052,0.9172,0.0074,0.9374,0.0054,3
4,InDiBias,grouped,BERT-CTX,base,collapsed,0.9366,0.0029,0.9161,0.0035,0.9364,0.0028,3
5,InDiBias,grouped,BERT-CTX,reg,collapsed,0.9397,0.0081,0.9199,0.0095,0.9394,0.0076,3
6,InDiBias,grouped,SBERT,base,raw,0.9407,0.0126,0.9224,0.0151,0.9409,0.0120,3
7,InDiBias,grouped,SBERT,reg,raw,0.9376,0.0014,0.9181,0.0024,0.9377,0.0016,3
8,InDiBias,grouped,BERT-ISO,base,raw,0.9356,0.0050,0.9144,0.0064,0.9353,0.0049,3
9,InDiBias,grouped,BERT-ISO,reg,raw,0.9376,0.0052,0.9172,0.0074,0.9374,0.0054,3


Saved results/rgat_repeat_results_base_reg_raw_collapsed.csv and results/rgat_accuracy_f1_results_base_reg_raw_collapsed.csv


## 6. Download the result tables

The summary file contains the mean and standard deviation of accuracy, macro-F1 and weighted-F1 across repeats. The repeat-level file contains each repeat's metrics.


In [ ]:
# Optional: download the summary CSV directly from Google Colab.
try:
    from google.colab import files
    files.download("results/rgat_accuracy_f1_results.csv")
except ImportError:
    print("Not running in Google Colab. Find the file at results/rgat_accuracy_f1_results.csv")


FileNotFoundError: Cannot find file: results/rgat_accuracy_f1_results.csv

## Reading the results

- **Accuracy**: fraction of correctly classified sentences.
- **Macro-F1**: F1 computed per class and averaged equally; useful when stereotype and anti-stereotype counts differ.
- **Weighted-F1**: class F1 scores averaged according to class support.
- CV is plain stratified 5-fold on both datasets. A grouped split was considered to protect InDiBias minimal pairs, but the annotation files carry no `template_id`, so it offered no real protection over stratified and was dropped.
- Compare against the existing R-GCN/GAT results only when dataset versions, folds, encoder features, graph conversion and relation handling match. This notebook uses collapsed relation families by default.


# Running confirmatory tests